# Getting Started: VarQITE for H2

This notebook runs a minimal **Variational Quantum Imaginary-Time Evolution
(VarQITE)** workflow for the hydrogen molecule **H2**.

Goals:

- run a small VarQITE calculation
- inspect the energy trajectory
- compare the final result with the exact ground-state energy
- reconstruct the final statevector from the stored output

This is the smallest end-to-end imaginary-time workflow in the repository.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from common.hamiltonian import get_exact_spectrum
from qite.core import run_qite

## Why VarQITE?

**VarQITE** evolves a variational state in imaginary time:

$$
|\psi(\tau)\rangle \propto e^{-\tau H} |\psi(0)\rangle
$$

Imaginary-time evolution suppresses higher-energy components, so the state is
driven toward the ground state when the ansatz is expressive enough.

In this repository, VarQITE is implemented through a McLachlan-projected
variational update.

## Exact ground-state reference

For a small system like `H2`, we can compare the final VarQITE energy against
the exact ground-state value.

In [ ]:
exact_spectrum = np.asarray(get_exact_spectrum("H2"), dtype=float)
exact_spectrum = np.sort(exact_spectrum)

exact_ground_energy = float(exact_spectrum[0])
exact_ground_energy

## Run a minimal VarQITE calculation

We use:

- molecule: `H2`
- ansatz: `UCCSD`
- a modest number of imaginary-time steps
- a small imaginary-time step size `dtau`

VarQITE in this package is a pure-state workflow, so noisy simulation is not
used here.

In [ ]:
res = run_qite(
    molecule="H2",
    ansatz_name="UCCSD",
    steps=50,
    dtau=0.2,
    seed=0,
    noisy=False,
    force=True,
    plot=False,
)

In [ ]:
sorted(res.keys())

## Final energy

The main quantity of interest is the final variational energy after the last
imaginary-time update.

In [ ]:
final_energy = float(res["energy"])
final_energy

## Energy trajectory

VarQITE returns the full energy trace, including the initial point.

In [ ]:
energies = np.asarray(res["energies"], dtype=float)
energies

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(energies)), energies, marker="o")
plt.axhline(exact_ground_energy, linestyle="--", label="Exact ground energy")
plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title("VarQITE energy trajectory for H2")
plt.grid(True)
plt.legend()
plt.show()

## Compare against the exact ground-state energy

In [ ]:
abs_error = abs(final_energy - exact_ground_energy)

print(f"VarQITE final energy : {final_energy:.10f}")
print(f"Exact ground energy  : {exact_ground_energy:.10f}")
print(f"Absolute error       : {abs_error:.6e}")

## Final-state reconstruction

The result stores the final state as separate real and imaginary components.
We reconstruct the statevector and inspect the dominant basis amplitudes.

In [ ]:
psi_real = np.asarray(res["final_state_real"], dtype=float)
psi_imag = np.asarray(res["final_state_imag"], dtype=float)
psi = psi_real + 1j * psi_imag

num_qubits = int(res["num_qubits"])
num_qubits

In [ ]:
threshold = 1e-2
idx = np.where(np.abs(psi) > threshold)[0]

print(f"Significant components (|c_i| > {threshold:g}):")
for i in idx:
    amp = psi[i]
    bitstring = format(i, f"0{num_qubits}b")
    print(f"|{bitstring}> : {amp.real:.6f}{amp.imag:+.6f}j   |c_i|={abs(amp):.6f}")

## Amplitude magnitudes

A small bar plot makes the dominant computational-basis support easier to see.

In [ ]:
labels = [f"|{i:0{num_qubits}b}>" for i in idx]
magnitudes = np.abs(psi[idx])

plt.figure(figsize=(8, 4))
plt.bar(labels, magnitudes)
plt.xlabel("Computational basis state")
plt.ylabel(r"$|c_i|$")
plt.title("Final-state amplitudes from VarQITE for H2")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

## Interpretation

A useful way to think about the methods in this repository is:

- **VQE** minimizes an energy expectation value directly
- **VarQITE** follows an imaginary-time projected update
- **QPE** extracts eigenvalue information from phase evolution

All three workflows use the same Hamiltonian pipeline, so comparisons across
methods are physically consistent.

## What this notebook showed

We:

- ran a minimal VarQITE workflow for `H2`
- inspected the returned energy trajectory
- compared the final energy with the exact ground-state reference
- reconstructed the final statevector from the stored output

This is the basic imaginary-time workflow in the repository.

## Next steps

Good follow-ups are:

- vary `dtau` and the number of steps
- compare `VarQITE` and `VQE` on the same molecule
- compare different ansätze under the same imaginary-time settings